<a href="https://colab.research.google.com/github/shreya-ds101/identity-access-risk-analysis/blob/main/01_project_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/shreya-ds101/identity-access-risk-analysis"

Cloning into 'identity-access-risk-analysis'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [3]:
%cd identity-access-risk-analysis
!ls

/content/identity-access-risk-analysis
README.md


In [4]:
from pathlib import Path

folders = [
    "data/raw",
    "data/processed",
    "src",
    "notebooks",
    "powerbi"
]

for folder in folders:
    path = Path(folder)
    path.mkdir(parents=True, exist_ok=True)
    (path / ".gitkeep").touch(exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


Create the employee dataset
In this step, we create a synthetic employee master dataset for the Identity and Access Management (IAM) project.



In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# Fixed seed makes the same data reproducible
rng = np.random.default_rng(42)

reference_date = pd.Timestamp("2026-11-06")
start_date = pd.Timestamp("2018-01-01")

department_roles = {
    "HR": ["HR Executive", "Recruiter", "HR Manager"],
    "Finance": ["Accountant", "Financial Analyst", "Finance Manager"],
    "Marketing": ["Marketing Executive", "Marketing Manager"],
    "Sales": ["Sales Executive", "Sales Manager"],
    "IT": ["IT Support", "System Administrator", "IT Manager"],
    "Operations": ["Operations Analyst", "Operations Manager"],
    "Customer Support": ["Support Agent", "Support Manager"],
    "Legal & Compliance": ["Compliance Analyst", "Compliance Manager"]
}

locations = [
    "Delhi", "Mumbai", "Bengaluru",
    "Hyderabad", "Pune", "Chennai"
]

employees = []

for number in range(1, 5241):
    department = rng.choice(list(department_roles.keys()))
    job_role = rng.choice(department_roles[department])

    random_days = int(
        rng.integers(
            0,
            (reference_date - start_date).days + 1
        )
    )

    join_date = start_date + pd.Timedelta(days=random_days)

    employee = {
        "employee_id": f"E{number:04d}",
        "employee_name": f"Employee {number:04d}",
        "department": department,
        "job_role": job_role,
        "manager_id": f"M{int(rng.integers(1, 301)):03d}",
        "location": rng.choice(locations),
        "join_date": join_date.date(),
        "employment_status": rng.choice(
            ["Active", "On Leave", "Inactive"],
            p=[0.95, 0.03, 0.02]
        )
    }

    employees.append(employee)

employees_df = pd.DataFrame(employees)

# Save the dataset
Path("data/raw").mkdir(parents=True, exist_ok=True)
employees_df.to_csv("data/raw/employees.csv", index=False)

print("Employee dataset created successfully.")
print("Dataset shape:", employees_df.shape)
display(employees_df.head())

Employee dataset created successfully.
Dataset shape: (5240, 8)


,employee_id,employee_name,department,job_role,manager_id,location,join_date,employment_status
0,E0001,Employee 0001,HR,HR Manager,M132,Bengaluru,2023-10-17,Active
1,E0002,Employee 0002,Customer Support,Support Agent,M158,Chennai,2018-11-01,Active
2,E0003,Employee 0003,Operations,Operations Manager,M039,Chennai,2022-07-17,Active
3,E0004,Employee 0004,Sales,Sales Executive,M235,Hyderabad,2026-03-15,Active
4,E0005,Employee 0005,IT,System Administrator,M069,Delhi,2021-12-26,Active


#Application and Permission Catalog

This step creates a standardized catalog of organizational applications and their available permissions. Each application is classified by business function, owner department, and criticality. Each permission is assigned an access level and sensitivity score for subsequent access-risk analysis.

The generated files are:

- `data/raw/applications.csv`
- `data/raw/permissions.csv`

These files will later be used to assign permissions to employees and identify excessive or high-impact access.

In [6]:
import pandas as pd
from pathlib import Path

# Application catalog
applications_data = [
    {
        "application_id": "APP001",
        "application_name": "HR Portal",
        "category": "Human Resources",
        "owner_department": "HR",
        "criticality": "High"
    },
    {
        "application_id": "APP002",
        "application_name": "Payroll System",
        "category": "Finance",
        "owner_department": "Finance",
        "criticality": "Critical"
    },
    {
        "application_id": "APP003",
        "application_name": "Finance Database",
        "category": "Finance",
        "owner_department": "Finance",
        "criticality": "Critical"
    },
    {
        "application_id": "APP004",
        "application_name": "Customer CRM",
        "category": "Customer Management",
        "owner_department": "Sales",
        "criticality": "High"
    },
    {
        "application_id": "APP005",
        "application_name": "Marketing Analytics",
        "category": "Analytics",
        "owner_department": "Marketing",
        "criticality": "Medium"
    },
    {
        "application_id": "APP006",
        "application_name": "Customer Support Desk",
        "category": "Customer Support",
        "owner_department": "Customer Support",
        "criticality": "High"
    },
    {
        "application_id": "APP007",
        "application_name": "IT Service Management",
        "category": "IT Operations",
        "owner_department": "IT",
        "criticality": "High"
    },
    {
        "application_id": "APP008",
        "application_name": "Admin Console",
        "category": "System Administration",
        "owner_department": "IT",
        "criticality": "Critical"
    },
    {
        "application_id": "APP009",
        "application_name": "Data Warehouse",
        "category": "Data and Reporting",
        "owner_department": "IT",
        "criticality": "Critical"
    },
    {
        "application_id": "APP010",
        "application_name": "Legal Case Management",
        "category": "Legal",
        "owner_department": "Legal & Compliance",
        "criticality": "High"
    },
    {
        "application_id": "APP011",
        "application_name": "Operations Dashboard",
        "category": "Operations",
        "owner_department": "Operations",
        "criticality": "Medium"
    },
    {
        "application_id": "APP012",
        "application_name": "Email and Collaboration",
        "category": "Communication",
        "owner_department": "IT",
        "criticality": "High"
    }
]

applications_df = pd.DataFrame(applications_data)

# Permission definitions for each application
permission_templates = {
    "APP001": [
        ("View Employee Profile", "Read"),
        ("Edit Employee Profile", "Write"),
        ("Approve Leave", "Approve"),
        ("Manage HR Settings", "Admin")
    ],
    "APP002": [
        ("View Payroll Records", "Read"),
        ("Process Payroll", "Write"),
        ("Export Payroll Data", "Export"),
        ("Administer Payroll", "Admin")
    ],
    "APP003": [
        ("Read Financial Records", "Read"),
        ("Edit Financial Records", "Write"),
        ("Approve Payments", "Approve"),
        ("Export Financial Data", "Export"),
        ("Database Administration", "Admin")
    ],
    "APP004": [
        ("View Customer Records", "Read"),
        ("Edit Customer Records", "Write"),
        ("Export Customer Data", "Export"),
        ("CRM Administration", "Admin")
    ],
    "APP005": [
        ("View Campaign Data", "Read"),
        ("Edit Campaign Data", "Write"),
        ("Export Marketing Reports", "Export"),
        ("Analytics Administration", "Admin")
    ],
    "APP006": [
        ("View Support Tickets", "Read"),
        ("Update Support Tickets", "Write"),
        ("Export Customer Tickets", "Export"),
        ("Support Administration", "Admin")
    ],
    "APP007": [
        ("View IT Tickets", "Read"),
        ("Resolve IT Tickets", "Write"),
        ("Manage Devices", "Approve"),
        ("ITSM Administration", "Admin")
    ],
    "APP008": [
        ("View System Logs", "Read"),
        ("Manage Users", "Write"),
        ("Grant Permissions", "Approve"),
        ("System Administration", "Admin")
    ],
    "APP009": [
        ("Read Warehouse Data", "Read"),
        ("Write Warehouse Data", "Write"),
        ("Export Warehouse Data", "Export"),
        ("Warehouse Administration", "Admin")
    ],
    "APP010": [
        ("View Legal Cases", "Read"),
        ("Edit Legal Cases", "Write"),
        ("Export Legal Reports", "Export"),
        ("Legal Administration", "Admin")
    ],
    "APP011": [
        ("View Operations Metrics", "Read"),
        ("Edit Operations Metrics", "Write"),
        ("Approve Operational Changes", "Approve"),
        ("Operations Administration", "Admin")
    ],
    "APP012": [
        ("View Email Metadata", "Read"),
        ("Manage Collaboration Groups", "Write"),
        ("Collaboration Administration", "Admin")
    ]
}

# Sensitivity score for later risk calculations
sensitivity_scores = {
    "Read": 1,
    "Write": 2,
    "Approve": 3,
    "Export": 4,
    "Admin": 5
}

permission_rows = []

for application_id, permission_list in permission_templates.items():
    for permission_name, permission_level in permission_list:
        permission_rows.append({
            "permission_id": f"PERM{len(permission_rows) + 1:03d}",
            "application_id": application_id,
            "permission_name": permission_name,
            "permission_level": permission_level,
            "sensitivity_score": sensitivity_scores[permission_level]
        })

permissions_df = pd.DataFrame(permission_rows)

# Save both files
Path("data/raw").mkdir(parents=True, exist_ok=True)

applications_df.to_csv(
    "data/raw/applications.csv",
    index=False
)

permissions_df.to_csv(
    "data/raw/permissions.csv",
    index=False
)

print("Application and permission catalogs created successfully.")
print("Applications:", applications_df.shape)
print("Permissions:", permissions_df.shape)

display(applications_df.head())
display(permissions_df.head())

Application and permission catalogs created successfully.
Applications: (12, 5)
Permissions: (48, 5)


,application_id,application_name,category,owner_department,criticality
0,APP001,HR Portal,Human Resources,HR,High
1,APP002,Payroll System,Finance,Finance,Critical
2,APP003,Finance Database,Finance,Finance,Critical
3,APP004,Customer CRM,Customer Management,Sales,High
4,APP005,Marketing Analytics,Analytics,Marketing,Medium


,permission_id,application_id,permission_name,permission_level,sensitivity_score
0,PERM001,APP001,View Employee Profile,Read,1
1,PERM002,APP001,Edit Employee Profile,Write,2
2,PERM003,APP001,Approve Leave,Approve,3
3,PERM004,APP001,Manage HR Settings,Admin,5
4,PERM005,APP002,View Payroll Records,Read,1


# Employee Access Assignment

This step creates the relationship between employees and the permissions they currently hold.

Most employees receive role-appropriate access. A controlled number of employees also receive cross-department, administrative, or temporary permissions to simulate real-world access-risk scenarios.

The generated access records will be saved as:

`data/raw/access_permissions.csv`

This file will later be used to detect excessive access, privilege creep, and role mismatches.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducible random generator
rng = np.random.default_rng(42)

reference_date = pd.Timestamp("2026-11-06")

# Load previously created datasets
employees_df = pd.read_csv(
    "data/raw/employees.csv",
    parse_dates=["join_date"]
)

applications_df = pd.read_csv(
    "data/raw/applications.csv"
)

permissions_df = pd.read_csv(
    "data/raw/permissions.csv"
)

# Applications normally required by each department
allowed_applications = {
    "HR": ["APP001", "APP012"],
    "Finance": ["APP002", "APP003", "APP009", "APP012"],
    "Marketing": ["APP005", "APP012"],
    "Sales": ["APP004", "APP012"],
    "IT": ["APP007", "APP008", "APP009", "APP012"],
    "Operations": ["APP011", "APP012"],
    "Customer Support": ["APP004", "APP006", "APP012"],
    "Legal & Compliance": ["APP010", "APP012"]
}

# Create lookups for faster processing
permission_lookup = permissions_df.set_index(
    "permission_id"
).to_dict("index")

# Permission IDs that can create higher-risk scenarios
risk_permission_pool = permissions_df[
    (
        permissions_df["permission_level"].isin(
            ["Approve", "Export", "Admin"]
        )
    )
    &
    (permissions_df["application_id"] != "APP012")
]["permission_id"].tolist()

# Select 428 employees for controlled risk scenarios
risky_employee_ids = set(
    rng.choice(
        employees_df["employee_id"].tolist(),
        size=428,
        replace=False
    )
)

access_records = []
employee_permission_map = {}

for _, employee in employees_df.iterrows():

    employee_id = employee["employee_id"]
    department = employee["department"]

    # Find permissions belonging to the employee's normal applications
    normal_permission_pool = permissions_df[
        permissions_df["application_id"].isin(
            allowed_applications[department]
        )
    ]["permission_id"].tolist()

    # Give each employee six normal permissions
    selected_permissions = rng.choice(
        normal_permission_pool,
        size=6,
        replace=False
    ).tolist()

    # Inject one unrelated high-impact permission for selected employees
    if employee_id in risky_employee_ids:

        unrelated_permissions = [
            permission_id
            for permission_id in risk_permission_pool
            if permission_id not in selected_permissions
        ]

        selected_permissions[-1] = rng.choice(
            unrelated_permissions
        )

    employee_permission_map[employee_id] = set(
        selected_permissions
    )

    for permission_id in selected_permissions:

        permission_details = permission_lookup[permission_id]
        application_id = permission_details["application_id"]

        is_role_aligned = (
            application_id in allowed_applications[department]
        )

        if is_role_aligned:
            access_source = "Role-based"
            is_temporary = False
        else:
            access_source = rng.choice(
                ["Manager-approved", "Temporary project"]
            )
            is_temporary = access_source == "Temporary project"

        join_date = pd.Timestamp(employee["join_date"])
        maximum_days = max(
            0,
            (reference_date - join_date).days
        )

        granted_date = join_date + pd.Timedelta(
            days=int(rng.integers(0, maximum_days + 1))
        )

        expiry_date = None

        if is_temporary:
            expiry_date = granted_date + pd.Timedelta(
                days=int(rng.integers(30, 181))
            )

        access_records.append({
            "employee_id": employee_id,
            "permission_id": permission_id,
            "application_id": application_id,
            "access_granted_date": granted_date.date(),
            "access_source": access_source,
            "is_temporary": is_temporary,
            "access_expiry_date": (
                expiry_date.date()
                if expiry_date is not None
                else None
            ),
            "access_status": "Active"
        })

# Add 10 additional access records to create variation
extra_employees = list(risky_employee_ids)[:10]

for employee_id in extra_employees:

    employee = employees_df[
        employees_df["employee_id"] == employee_id
    ].iloc[0]

    unused_permissions = [
        permission_id
        for permission_id in permissions_df["permission_id"]
        if permission_id not in employee_permission_map[employee_id]
    ]

    permission_id = rng.choice(unused_permissions)
    permission_details = permission_lookup[permission_id]

    join_date = pd.Timestamp(employee["join_date"])
    maximum_days = max(
        0,
        (reference_date - join_date).days
    )

    granted_date = join_date + pd.Timedelta(
        days=int(rng.integers(0, maximum_days + 1))
    )

    access_records.append({
        "employee_id": employee_id,
        "permission_id": permission_id,
        "application_id": permission_details["application_id"],
        "access_granted_date": granted_date.date(),
        "access_source": "Manager-approved",
        "is_temporary": False,
        "access_expiry_date": None,
        "access_status": "Active"
    })

# Convert to DataFrame
access_permissions_df = pd.DataFrame(access_records)

# Add unique access IDs
access_permissions_df.insert(
    0,
    "access_id",
    [
        f"ACC{number:05d}"
        for number in range(
            1,
            len(access_permissions_df) + 1
        )
    ]
)

# Save the access assignment dataset
Path("data/raw").mkdir(parents=True, exist_ok=True)

access_permissions_df.to_csv(
    "data/raw/access_permissions.csv",
    index=False
)

print("Employee access dataset created successfully.")
print("Dataset shape:", access_permissions_df.shape)
print(
    "Non-role-based access records:",
    (access_permissions_df["access_source"] != "Role-based").sum()
)

display(access_permissions_df.head())

Employee access dataset created successfully.
Dataset shape: (31450, 9)
Non-role-based access records: 422


,access_id,employee_id,permission_id,application_id,access_granted_date,access_source,is_temporary,access_expiry_date,access_status
0,ACC00001,E0001,PERM047,APP012,2026-04-19,Role-based,False,None,Active
1,ACC00002,E0001,PERM001,APP001,2026-05-10,Role-based,False,None,Active
2,ACC00003,E0001,PERM004,APP001,2025-02-26,Role-based,False,None,Active
3,ACC00004,E0001,PERM046,APP012,2023-11-30,Role-based,False,None,Active
4,ACC00005,E0001,PERM002,APP001,2025-03-09,Role-based,False,None,Active


#Access Activity Logs

This step simulates employee activity across the applications they can access.

The logs include access timestamps, login locations, actions performed, and after-hours activity. Some assigned permissions intentionally have no activity so that dormant access can be identified later.

The generated file will be saved as:

`data/raw/access_logs.csv`

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

rng = np.random.default_rng(42)
reference_date = pd.Timestamp("2026-11-06")

# Load previously created datasets
employees_df = pd.read_csv(
    "data/raw/employees.csv"
)

access_permissions_df = pd.read_csv(
    "data/raw/access_permissions.csv",
    parse_dates=["access_granted_date"]
)

permissions_df = pd.read_csv(
    "data/raw/permissions.csv"
)

# Create lookup dictionaries
employee_lookup = employees_df.set_index(
    "employee_id"
).to_dict("index")

permission_lookup = permissions_df.set_index(
    "permission_id"
).to_dict("index")

all_locations = employees_df["location"].unique().tolist()

# Employees with non-role-based access are more likely to show unusual activity
attention_employees = set(
    access_permissions_df[
        access_permissions_df["access_source"] != "Role-based"
    ]["employee_id"].unique()
)

access_logs = []

for _, access in access_permissions_df.iterrows():

    employee_id = access["employee_id"]
    employee = employee_lookup[employee_id]
    permission = permission_lookup[access["permission_id"]]

    # Leave around 28% of permissions unused to simulate dormant access
    if rng.random() > 0.72:
        continue

    granted_date = pd.Timestamp(
        access["access_granted_date"]
    )

    maximum_days = max(
        0,
        (reference_date - granted_date).days
    )

    random_days = int(
        rng.integers(0, maximum_days + 1)
    )

    activity_date = granted_date + pd.Timedelta(
        days=random_days
    )

    # Higher chance of unusual behaviour for attention employees
    unusual_activity = (
        employee_id in attention_employees
        and rng.random() < 0.35
    )

    if unusual_activity:

        unusual_locations = [
            location
            for location in all_locations
            if location != employee["location"]
        ]

        login_location = rng.choice(unusual_locations)

        hour = int(
            rng.choice(
                [0, 1, 2, 3, 4, 5, 6, 7,
                 19, 20, 21, 22, 23]
            )
        )

    else:

        login_location = employee["location"]
        hour = int(rng.integers(8, 19))

    minute = int(rng.integers(0, 60))
    second = int(rng.integers(0, 60))

    activity_timestamp = activity_date.replace(
        hour=hour,
        minute=minute,
        second=second
    )

    permission_level = permission["permission_level"]

    activity_type = {
        "Read": "View",
        "Write": "Update",
        "Approve": "Approve",
        "Export": "Export",
        "Admin": "Administrative action"
    }[permission_level]

    access_logs.append({
        "employee_id": employee_id,
        "access_id": access["access_id"],
        "application_id": access["application_id"],
        "permission_id": access["permission_id"],
        "activity_timestamp": activity_timestamp,
        "login_location": login_location,
        "activity_type": activity_type,
        "access_result": (
            "Failed"
            if rng.random() < 0.03
            else "Success"
        ),
        "is_after_hours": (
            hour < 8 or hour >= 19
        )
    })

# Convert to DataFrame
access_logs_df = pd.DataFrame(access_logs)

# Add unique log IDs
access_logs_df.insert(
    0,
    "log_id",
    [
        f"LOG{number:06d}"
        for number in range(
            1,
            len(access_logs_df) + 1
        )
    ]
)

# Save the activity logs
Path("data/raw").mkdir(parents=True, exist_ok=True)

access_logs_df.to_csv(
    "data/raw/access_logs.csv",
    index=False
)

print("Access activity logs created successfully.")
print("Dataset shape:", access_logs_df.shape)
print(
    "After-hours events:",
    access_logs_df["is_after_hours"].sum()
)
print(
    "Failed access attempts:",
    (access_logs_df["access_result"] == "Failed").sum()
)

display(access_logs_df.head())

Access activity logs created successfully.
Dataset shape: (22591, 10)
After-hours events: 648
Failed access attempts: 676


,log_id,employee_id,access_id,application_id,permission_id,activity_timestamp,login_location,activity_type,access_result,is_after_hours
0,LOG000001,E0001,ACC00002,APP001,PERM001,2026-07-27 17:05:41,Bengaluru,View,Success,False
1,LOG000002,E0001,ACC00006,APP012,PERM048,2026-06-17 12:30:22,Bengaluru,Administrative action,Success,False
2,LOG000003,E0002,ACC00007,APP006,PERM024,2022-10-20 17:32:26,Chennai,Export,Success,False
3,LOG000004,E0002,ACC00008,APP012,PERM047,2026-09-06 08:51:49,Chennai,Update,Success,False
4,LOG000005,E0002,ACC00010,APP004,PERM014,2019-10-10 18:26:53,Chennai,View,Success,False


#Rule-Based Access Risk Scoring

This step combines employee, permission, access, and activity data to calculate an explainable risk score for each employee.

The score considers role-mismatched access, dormant permissions, high-impact permissions, after-hours activity, unusual locations, and failed access attempts.

Each employee is assigned a risk level:

- Low
- Medium
- High

The results will be saved as:

- `data/processed/risk_assessments.csv`
- `data/processed/access_analysis.csv`

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load project datasets
employees_df = pd.read_csv(
    "data/raw/employees.csv"
)

permissions_df = pd.read_csv(
    "data/raw/permissions.csv"
)

access_permissions_df = pd.read_csv(
    "data/raw/access_permissions.csv"
)

access_logs_df = pd.read_csv(
    "data/raw/access_logs.csv"
)

# Applications normally required by each department
allowed_applications = {
    "HR": ["APP001", "APP012"],
    "Finance": ["APP002", "APP003", "APP009", "APP012"],
    "Marketing": ["APP005", "APP012"],
    "Sales": ["APP004", "APP012"],
    "IT": ["APP007", "APP008", "APP009", "APP012"],
    "Operations": ["APP011", "APP012"],
    "Customer Support": ["APP004", "APP006", "APP012"],
    "Legal & Compliance": ["APP010", "APP012"]
}

# Combine access records with employee and permission details
access_analysis_df = access_permissions_df.merge(
    employees_df[
        ["employee_id", "department", "location"]
    ],
    on="employee_id",
    how="left"
)

access_analysis_df = access_analysis_df.merge(
    permissions_df[
        [
            "permission_id",
            "permission_name",
            "permission_level"
        ]
    ],
    on="permission_id",
    how="left"
)

# Identify role-mismatched permissions
access_analysis_df["role_aligned"] = [
    application_id in allowed_applications[department]
    for application_id, department
    in zip(
        access_analysis_df["application_id"],
        access_analysis_df["department"]
    )
]

# Identify permissions with no successful usage
successful_access_ids = set(
    access_logs_df[
        access_logs_df["access_result"] == "Success"
    ]["access_id"]
)

access_analysis_df["is_dormant"] = (
    ~access_analysis_df["access_id"].isin(
        successful_access_ids
    )
)

# Summarize permission-related risk factors
permission_summary_df = access_analysis_df.groupby(
    "employee_id"
).agg(
    total_permissions=("permission_id", "nunique"),
    dormant_permissions=("is_dormant", "sum"),
    cross_department_permissions=(
        "role_aligned",
        lambda values: (~values).sum()
    ),
    admin_permissions=(
        "permission_level",
        lambda values: (values == "Admin").sum()
    ),
    high_impact_permissions=(
        "permission_level",
        lambda values: values.isin(
            ["Admin", "Approve", "Export"]
        ).sum()
    )
).reset_index()

# Identify unusual activity
logs_analysis_df = access_logs_df.merge(
    employees_df[
        ["employee_id", "location"]
    ],
    on="employee_id",
    how="left"
)

logs_analysis_df["unusual_location"] = (
    logs_analysis_df["login_location"]
    != logs_analysis_df["location"]
)

# Summarize activity-related risk factors
activity_summary_df = logs_analysis_df.groupby(
    "employee_id"
).agg(
    after_hours_events=("is_after_hours", "sum"),
    unusual_location_events=("unusual_location", "sum"),
    failed_access_attempts=(
        "access_result",
        lambda values: (values == "Failed").sum()
    ),
    successful_events=(
        "access_result",
        lambda values: (values == "Success").sum()
    )
).reset_index()

# Create one employee-level risk table
risk_df = employees_df.merge(
    permission_summary_df,
    on="employee_id",
    how="left"
).merge(
    activity_summary_df,
    on="employee_id",
    how="left"
)

# Replace missing values with zero
numeric_columns = [
    "total_permissions",
    "dormant_permissions",
    "cross_department_permissions",
    "admin_permissions",
    "high_impact_permissions",
    "after_hours_events",
    "unusual_location_events",
    "failed_access_attempts",
    "successful_events"
]

risk_df[numeric_columns] = risk_df[numeric_columns].fillna(0)

# Calculate transparent risk points
risk_df["dormant_access_points"] = (
    risk_df["dormant_permissions"] * 5
).clip(upper=20)

risk_df["cross_department_points"] = (
    risk_df["cross_department_permissions"] * 20
).clip(upper=30)

risk_df["admin_permission_points"] = (
    risk_df["admin_permissions"] * 20
).clip(upper=40)

other_high_impact_permissions = (
    risk_df["high_impact_permissions"]
    - risk_df["admin_permissions"]
).clip(lower=0)

risk_df["high_impact_points"] = (
    other_high_impact_permissions * 8
).clip(upper=16)

risk_df["after_hours_points"] = (
    risk_df["after_hours_events"] * 5
).clip(upper=15)

risk_df["unusual_location_points"] = (
    risk_df["unusual_location_events"] * 10
).clip(upper=20)

risk_df["failed_attempt_points"] = (
    risk_df["failed_access_attempts"] * 5
).clip(upper=10)

risk_df["excessive_permission_points"] = np.where(
    risk_df["total_permissions"] > 6,
    5,
    0
)

# Calculate final risk score
risk_df["risk_score"] = (
    risk_df["dormant_access_points"]
    + risk_df["cross_department_points"]
    + risk_df["admin_permission_points"]
    + risk_df["high_impact_points"]
    + risk_df["after_hours_points"]
    + risk_df["unusual_location_points"]
    + risk_df["failed_attempt_points"]
    + risk_df["excessive_permission_points"]
).clip(upper=100).astype(int)

# Assign risk level
risk_df["risk_level"] = np.select(
    [
        risk_df["risk_score"] >= 60,
        risk_df["risk_score"] >= 30
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

# Generate an explanation for every employee
def create_reason(row):
    reasons = []

    if row["cross_department_permissions"] > 0:
        reasons.append(
            "Role-mismatched access"
        )

    if row["dormant_permissions"] > 0:
        reasons.append(
            "Dormant permissions"
        )

    if row["admin_permissions"] > 0:
        reasons.append(
            "Administrative privilege"
        )

    if row["unusual_location_events"] > 0:
        reasons.append(
            "Unusual login location"
        )

    if row["after_hours_events"] > 0:
        reasons.append(
            "After-hours activity"
        )

    if row["failed_access_attempts"] > 0:
        reasons.append(
            "Failed access attempts"
        )

    if not reasons:
        reasons.append(
            "No significant risk indicators"
        )

    return "; ".join(reasons)

risk_df["main_reason"] = risk_df.apply(
    create_reason,
    axis=1
)

# Save processed outputs
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

risk_df.to_csv(
    "data/processed/risk_assessments.csv",
    index=False
)

access_analysis_df.to_csv(
    "data/processed/access_analysis.csv",
    index=False
)

print("Risk assessment completed successfully.")
print("Employee risk records:", risk_df.shape[0])
print("\nRisk-level distribution:")
print(risk_df["risk_level"].value_counts())

display(
    risk_df.sort_values(
        "risk_score",
        ascending=False
    )[
        [
            "employee_id",
            "department",
            "job_role",
            "risk_score",
            "risk_level",
            "main_reason"
        ]
    ].head(10)
)

Risk assessment completed successfully.
Employee risk records: 5240

Risk-level distribution:
risk_level
Medium    3545
High      1403
Low        292
Name: count, dtype: int64


,employee_id,department,job_role,risk_score,risk_level,main_reason
1633,E1634,HR,Recruiter,100,High,Role-mismatched access; Dormant permissions; A...
3932,E3933,Legal & Compliance,Compliance Analyst,100,High,Role-mismatched access; Dormant permissions; A...
2836,E2837,Operations,Operations Manager,100,High,Role-mismatched access; Dormant permissions; A...
4695,E4696,Finance,Finance Manager,100,High,Role-mismatched access; Dormant permissions; A...
4638,E4639,IT,System Administrator,100,High,Role-mismatched access; Dormant permissions; A...
1252,E1253,Marketing,Marketing Executive,100,High,Role-mismatched access; Administrative privile...
1250,E1251,Finance,Accountant,100,High,Role-mismatched access; Dormant permissions; A...
1199,E1200,IT,IT Support,100,High,Role-mismatched access; Dormant permissions; A...
1616,E1617,Operations,Operations Manager,100,High,Role-mismatched access; Dormant permissions; A...
3401,E3402,Sales,Sales Manager,100,High,Role-mismatched access; Dormant permissions; A...


#Anomaly Detection with Isolation Forest

This step uses Isolation Forest to identify employees whose access and activity patterns differ significantly from other employees.

The model considers permission volume, dormant access, role mismatches, administrative permissions, after-hours activity, unusual locations, and failed access attempts.

The machine-learning result complements the transparent rule-based risk score. The rule-based reasons will still explain why an employee was flagged.

The updated risk assessment file will contain:

- Machine-learning anomaly score
- Anomaly flag
- Combined risk score
- Combined risk level

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# Load the rule-based risk assessments
risk_df = pd.read_csv(
    "data/processed/risk_assessments.csv"
)

# Features used by the anomaly-detection model
feature_columns = [
    "total_permissions",
    "dormant_permissions",
    "cross_department_permissions",
    "admin_permissions",
    "high_impact_permissions",
    "after_hours_events",
    "unusual_location_events",
    "failed_access_attempts",
    "successful_events",
    "risk_score"
]

X = risk_df[feature_columns].fillna(0)

# Train the Isolation Forest model
isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

# -1 means anomaly and 1 means normal
risk_df["ml_anomaly_flag"] = isolation_model.fit_predict(X)

risk_df["is_anomalous"] = (
    risk_df["ml_anomaly_flag"] == -1
)

# Convert model output into a 0-100 anomaly score
raw_anomaly_scores = -isolation_model.decision_function(X)

minimum_score = raw_anomaly_scores.min()
maximum_score = raw_anomaly_scores.max()

if maximum_score > minimum_score:
    risk_df["anomaly_score"] = (
        (raw_anomaly_scores - minimum_score)
        / (maximum_score - minimum_score)
        * 100
    ).round().astype(int)
else:
    risk_df["anomaly_score"] = 0

# Combine transparent rules with machine-learning output
risk_df["combined_risk_score"] = (
    0.70 * risk_df["risk_score"]
    + 0.30 * risk_df["anomaly_score"]
).round().clip(upper=100).astype(int)

# Assign final risk levels
risk_df["combined_risk_level"] = np.select(
    [
        risk_df["combined_risk_score"] >= 60,
        risk_df["combined_risk_score"] >= 30
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

# Save the updated assessment file
risk_df.to_csv(
    "data/processed/risk_assessments.csv",
    index=False
)

print("Isolation Forest analysis completed.")
print(
    "Anomalous employees:",
    risk_df["is_anomalous"].sum()
)

print("\nFinal risk-level distribution:")
print(
    risk_df["combined_risk_level"].value_counts()
)

display(
    risk_df.sort_values(
        "combined_risk_score",
        ascending=False
    )[
        [
            "employee_id",
            "department",
            "risk_score",
            "anomaly_score",
            "combined_risk_score",
            "combined_risk_level",
            "main_reason"
        ]
    ].head(10)
)

Isolation Forest analysis completed.
Anomalous employees: 262

Final risk-level distribution:
combined_risk_level
Medium    3810
Low        877
High       553
Name: count, dtype: int64


,employee_id,department,risk_score,anomaly_score,combined_risk_score,combined_risk_level,main_reason
5096,E5097,IT,100,100,100,High,Role-mismatched access; Dormant permissions; A...
3982,E3983,Sales,100,93,98,High,Role-mismatched access; Administrative privile...
686,E0687,Finance,100,92,98,High,Role-mismatched access; Dormant permissions; A...
446,E0447,Sales,100,93,98,High,Role-mismatched access; Administrative privile...
2422,E2423,Sales,100,94,98,High,Role-mismatched access; Dormant permissions; A...
3679,E3680,Sales,100,95,98,High,Role-mismatched access; Administrative privile...
528,E0529,Marketing,100,95,98,High,Role-mismatched access; Dormant permissions; A...
1252,E1253,Marketing,100,92,98,High,Role-mismatched access; Administrative privile...
3835,E3836,Customer Support,100,92,98,High,Role-mismatched access; Dormant permissions; A...
4669,E4670,IT,100,95,98,High,Role-mismatched access; Dormant permissions; A...
